In [ ]:
# Optimization Part 1
# Matrix Multiplcation Example
# Kernel 1 - Naive
# Kernel 2 - Row Coallesced
# Kernel 3 - Shared Memory Chunked
# Kernel 4 - 1D Tiling



In [ ]:
%%writefile kernel-1-naive.py

"""
  load from mata -> uncoalsced
  load from matb -> coalcsced
  store to matc -> uncoalsced
"""

# Kernel 1
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b

@cuda.jit
def matmul (matA , matB , matC, m, n, k):
  row = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  col = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
  if row < m and col < n:
    matC[row, col] = float32(0.0)
    for i in range(k):
      matC[row, col] += matA[row, i] * matB[i, col]


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (ceil_div(m, threads_per_block[0]), ceil_div(n, threads_per_block[1]))

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#

Overwriting kernel-1-naive.py


In [ ]:
!ncu --set full -f -o kernel-1-naive-rep python3 kernel-1-naive.py

==PROF== Connected to process 19412 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 19412
==PROF== Report: /content/kernel-1-naive-rep.ncu-rep


In [ ]:
%%writefile kernel-2-coallesced.py

"""
  load from mata -> coalsced
  load from matb -> uncoalcsced
  store to matc -> coalsced
"""

# Kernel 2
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b

@cuda.jit
def matmul (matA , matB , matC, m, n, k):
  # NOTICE the SWAP col v/s row
  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
  if row < m and col < n:
    matC[row, col] = float32(0.0)
    for i in range(k):
      matC[row, col] += matA[row, i] * matB[i, col]


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (32, 32)
# NOTICE the SWAP n v/s m = col v/s row
blocks_per_grid =  (
    ceil_div(n, threads_per_block[0]),
    ceil_div(m, threads_per_block[1])
)

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#

Overwriting kernel-2-coallesced.py


In [ ]:
!ncu --set full -f -o kernel-2-coallesced-rep python3 kernel-2-coallesced.py

==PROF== Connected to process 27963 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 27963
==PROF== Report: /content/kernel-2-coallesced-rep.ncu-rep


In [1]:
# Kernel 3 - Chunked using shared memory

%%writefile kernel-3-chunked.py

"""
  load from mata -> coalsced
  load from matb -> uncoalcsced
  store to matc -> coalsced
"""

from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b
tile_dim = 32

@cuda.jit
def matmul (matA , matB , matC, m, n, k):


  ceil_div = lambda a,b : (a + b - 1) // b

  tileA = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=float32)
  tileB = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=float32)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y

  out_col = tx + cuda.blockIdx.x * cuda.blockDim.x
  out_row = ty + cuda.blockIdx.y * cuda.blockDim.y


  num_tiles = ceil_div(k,tile_dim)
  product = float32(0.0)

  for t in range(num_tiles):
    # Phase 1
    # threads within the block will read elements from matA and matB (gmem)
    # and store them in tileA and tileB
    tile_row = ty + t * tile_dim
    tile_col = tx + t * tile_dim
    if out_row < m and tile_col < k:
      tileA[ty, tx] = matA[out_row, tile_col]
    else :
      tileA[ty, tx] = 0.0

    if tile_row < k and out_col < n:
      tileB[ty, tx] = matB[tile_row,out_col]
    else :
      tileB[ty, tx] = 0.0

    cuda.syncthreads()

    # Phase 2
    # threads will compute dot product by reading elemtns from tileA and tileB (smem)
    for i in range(tile_dim):
        product += tileA[ty, i] * tileB[i, tx]

    cuda.syncthreads()

  if out_row < m and out_col < n:
    prnt(product)
    matC[out_row, out_col] = product


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (tile_dim,tile_dim)
# NOTICE the SWAP n v/s m = col v/s row
blocks_per_grid =  (
    ceil_div(n, threads_per_block[0]),
    ceil_div(m, threads_per_block[1])
)

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#



Writing kernel-3-chunked.py


In [ ]:
!ncu --set full -f -o kernel-3-chunked-rep python3 kernel-3-chunked.py

==PROF== Connected to process 2583 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 2583
==PROF== Report: /content/kernel-3-chunked-rep.ncu-rep


In [9]:
# Kernel 4 - 1D Tiling - use less redas on SMEM but more registers.

%%writefile kernel-4-1D-Tiling.py

"""
  load from mata -> coalsced
  load from matb -> uncoalcsced
  store to matc -> coalsced
"""

from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b
BM = 32
BN = 32
BK = 32
TM = 8 # Number of o/p elements per Thread

NT = (BM * BN ) // TM # Number of Threads within a block

@cuda.jit
def matmul (matA , matB , matC, m, n, k):
  tileA = cuda.shared.array(shape=(BM, BK), dtype=float32)
  tileB = cuda.shared.array(shape=(BK, BN), dtype=float32)

  tid = cuda.threadIdx.x
  col = tid % BN
  row = (tid // BN) *  TM

  block_row = cuda.blockIdx.y * BM
  block_col = cuda.blockIdx.x * BN

  product = cuda.local.array(TM, dtype = cp.float32) # Local variable in the register
  for t in range(TM):
    product[t] = float32(0.0)

  for  k0 in range(0,k,BK):
    # phase 1:
    # threads will load values from matA and matB
    # into tileA and tileB
    # gmem -> smem

    #phase 1: load values from matA into tileA
    for i in range(tid, BM * BK , NT):
      r = i // BK
      c = i % BK
      global_mem_row = block_row + r
      global_mem_col = k0 + c
      if global_mem_row < m and global_mem_col < k:
        tileA[r, c] = matA[global_mem_row, global_mem_col]
      else:
        tileA[r, c] = float32(0.0)

    # Phase1: Load values from matB into tileB
    for i in range(tid, BK * BN , NT):
      r = i // BN
      c = i % BN
      global_mem_row = k0 + r
      global_mem_col = block_col + c
      if global_mem_row < k and global_mem_col < n:
        tileB[r, c] = matB[global_mem_row, global_mem_col]
      else:
        tileB[r, c] = float32(0.0)

    cuda.syncthreads()

    # Phase2:
    for kk  in range(BK):
      b = tileB[kk,col]
      for t in range(TM):
        product[t] += tileA[row+t,kk] * b

    cuda.syncthreads()

  global_mem_col = block_col + col
  for t in range(TM):
    global_mem_row = block_row + row + t
    if global_mem_row < m and global_mem_col < n:
      matC[global_mem_row, global_mem_col] = product[t]


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (
     NT ,# We'll use only 1D indexing here ,
     1
)

blocks_per_grid =  (
    ceil_div(n, BN),
    ceil_div(m, BM)
)

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#



Overwriting kernel-4-1D-Tiling.py


In [10]:
!ncu --set full -f -o kernel-4-1D-Tiling-rep python3 kernel-4-1D-Tiling.py

==PROF== Connected to process 2801 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 2801
==PROF== Report: /content/kernel-4-1D-Tiling-rep.ncu-rep
